# **Download and filesystem creation**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create .kaggle directory and move kaggle.json
!mkdir -p ~/.kaggle
import os
kaggle_json = "/root/.kaggle/kaggle.json"
import shutil

if not os.path.exists(kaggle_json):

    from google.colab import files

    print("Upload your kaggle.json file:")
    uploaded = files.upload()

    if "kaggle.json" not in uploaded:
        raise RuntimeError("kaggle.json was not uploaded.")

    os.makedirs("/root/.kaggle", exist_ok=True)

    shutil.move(
        "kaggle.json",
        kaggle_json
    )

    os.chmod(kaggle_json, 0o600)
# Verify the Kaggle configuration
!chmod 600 ~/.kaggle/kaggle.json
!ls -l ~/.kaggle

Upload your kaggle.json file:


Saving kaggle.json to kaggle.json
total 4
-rw------- 1 root root 69 Aug 23 19:28 kaggle.json


In [ ]:
!pip install kaggle

In [ ]:
import os

# List of Kaggle dataset identifiers (replace with your actual datasets)
kaggle_datasets = [
    "bhargavkumar1729/upload-folder-sdxl-aiface-dataset",
    "bhargavkumar1729/stylegan3-pytorch-faces-15k",
    "bhargavkumar1729/stylegan2-ada-pytorch-faces-15k",
    "bhargavkumar1729/gan-real-faces-50k-ffhq"
]

# Directory to store the downloaded data
download_dir = "./kaggle_data"
os.makedirs(download_dir, exist_ok=True)

for dataset_identifier in kaggle_datasets:
    print(f"\n--- Processing dataset: {dataset_identifier} ---")

    # Download the dataset
    # The -p flag specifies the path to download to
    # The -q flag makes the download quiet (no progress bar)
    download_command = f"kaggle datasets download -d {dataset_identifier} -p {download_dir} -q"
    print(f"Downloading: {dataset_identifier}")
    os.system(download_command)

    # Find the downloaded zip file(s)
    # Kaggle downloads the dataset as a zip file named after the dataset identifier's last part (e.g., 'first-dataset-name.zip')
    # or sometimes as a single zip file for the entire dataset.
    # We'll look for all .zip files in the download directory to be safe.

    # Get the base name of the dataset to look for its zip file
    dataset_base_name = dataset_identifier.split('/')[-1]
    zip_file_name_guess = os.path.join(download_dir, f"{dataset_base_name}.zip")

    # If the guessed zip file exists, unzip it. Otherwise, look for other zips.
    if os.path.exists(zip_file_name_guess):
        zip_files_to_process = [zip_file_name_guess]
    else:
        # Fallback: find all zip files in the download directory
        zip_files_to_process = [os.path.join(download_dir, f) for f in os.listdir(download_dir) if f.endswith('.zip')]

    if not zip_files_to_process:
        print(f"No zip files found for {dataset_identifier} in {download_dir}. Skipping unzip and delete.")
        continue

    for zip_file_path in zip_files_to_process:
        print(f"Unzipping: {os.path.basename(zip_file_path)}")
        # Unzip the file into the download directory
        unzip_command = f"unzip -o {zip_file_path} -d {download_dir}" # -o to overwrite existing files without prompting
        os.system(unzip_command)

        # Delete the zipped file
        print(f"Deleting zipped file: {os.path.basename(zip_file_path)}")
        os.remove(zip_file_path)

print("\nAll datasets processed.")
!rm -rf ./kaggle_data/my_dataset_upload
# Optional: List the contents of the download directory to verify
print("\nContents of the download directory:")
!ls -Fh {download_dir}



--- Processing dataset: bhargavkumar1729/upload-folder-sdxl-aiface-dataset ---
Downloading: bhargavkumar1729/upload-folder-sdxl-aiface-dataset
Unzipping: upload-folder-sdxl-aiface-dataset.zip
Deleting zipped file: upload-folder-sdxl-aiface-dataset.zip

--- Processing dataset: bhargavkumar1729/stylegan3-pytorch-faces-15k ---
Downloading: bhargavkumar1729/stylegan3-pytorch-faces-15k
Unzipping: stylegan3-pytorch-faces-15k.zip
Deleting zipped file: stylegan3-pytorch-faces-15k.zip

--- Processing dataset: bhargavkumar1729/stylegan2-ada-pytorch-faces-15k ---
Downloading: bhargavkumar1729/stylegan2-ada-pytorch-faces-15k
Unzipping: stylegan2-ada-pytorch-faces-15k.zip
Deleting zipped file: stylegan2-ada-pytorch-faces-15k.zip

--- Processing dataset: bhargavkumar1729/gan-real-faces-50k-ffhq ---
Downloading: bhargavkumar1729/gan-real-faces-50k-ffhq
Unzipping: gan-real-faces-50k-ffhq.zip
Deleting zipped file: gan-real-faces-50k-ffhq.zip

All datasets processed.

Contents of the download directory

In [ ]:
import os
import shutil
import random
from glob import glob
SEED = 42

random.seed(SEED)
# Define the base project directory
base_dir = "Faces_Project"
dataset_dir = os.path.join(base_dir, "Dataset")

# Define target subdirectories
train_real_dir = os.path.join(dataset_dir, "train", "real")
train_fake_dir = os.path.join(dataset_dir, "train", "fake")
val_real_dir = os.path.join(dataset_dir, "val", "real")
val_fake_dir = os.path.join(dataset_dir, "val", "fake")

# Create all necessary directories
os.makedirs(train_real_dir, exist_ok=True)
os.makedirs(train_fake_dir, exist_ok=True)
os.makedirs(val_real_dir, exist_ok=True)
os.makedirs(val_fake_dir, exist_ok=True)

print("Created directory structure:")
print(f"- {train_real_dir}")
print(f"- {train_fake_dir}")
print(f"- {val_real_dir}")
print(f"- {val_fake_dir}")

Created directory structure:
- Faces_Project/Dataset/train/real
- Faces_Project/Dataset/train/fake
- Faces_Project/Dataset/val/real
- Faces_Project/Dataset/val/fake


In [ ]:
# Source directory for real images
real_source_dir = "./kaggle_data/real"

# List all real image files (assuming .jpg, .png, etc.)
real_images = []
for ext in ('*.jpg', '*.jpeg', '*.png'):
    real_images.extend(glob(os.path.join(real_source_dir, '**', ext), recursive=True))

print(f"Found {len(real_images)} real images.")

# Shuffle the images
random.shuffle(real_images)

# Define split counts
train_real_count = 40000
val_real_count = 10000

if len(real_images) < (train_real_count + val_real_count):
    print(f"Warning: Not enough real images ({len(real_images)}) for desired split (50000). Using all available.")
    train_real_count = int(len(real_images) * 0.8) # Adjust proportionally
    val_real_count = len(real_images) - train_real_count

# Split into train and validation sets
train_real_images = real_images[:train_real_count]
val_real_images = real_images[train_real_count : train_real_count + val_real_count]

print(f"Moving {len(train_real_images)} real images to training.")
for i, img_path in enumerate(train_real_images):
    new_name = f"real_{i:05d}{os.path.splitext(img_path)[1]}"
    shutil.move(img_path, os.path.join(train_real_dir, new_name))

print(f"Moving {len(val_real_images)} real images to validation.")
for i, img_path in enumerate(val_real_images):
    new_name = f"real_{i:05d}{os.path.splitext(img_path)[1]}"
    shutil.move(img_path, os.path.join(val_real_dir, new_name))

print("Real images processed.")

Found 50000 real images.
Moving 40000 real images to training.
Moving 10000 real images to validation.
Real images processed.


In [ ]:
# Source directories for fake images
fake_source_dirs = [
    "./kaggle_data/Final_SDXL_AIface_DS",
    "./kaggle_data/stylegan2_faces",
    "./kaggle_data/stylegan3_faces"
]

# List all fake image files
fake_images = []
for source_dir in fake_source_dirs:
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        fake_images.extend(glob(os.path.join(source_dir, '**', ext), recursive=True))

print(f"Found {len(fake_images)} fake images.")

# Shuffle the images
random.shuffle(fake_images)

# Define split counts
train_fake_count = 40000
val_fake_count = 10000

if len(fake_images) < (train_fake_count + val_fake_count):
    print(f"Warning: Not enough fake images ({len(fake_images)}) for desired split (50000). Using all available.")
    train_fake_count = int(len(fake_images) * 0.8) # Adjust proportionally
    val_fake_count = len(fake_images) - train_fake_count

# Split into train and validation sets
train_fake_images = fake_images[:train_fake_count]
val_fake_images = fake_images[train_fake_count : train_fake_count + val_fake_count]

print(f"Moving {len(train_fake_images)} fake images to training.")
for i, img_path in enumerate(train_fake_images):
    new_name = f"fake_{i:05d}{os.path.splitext(img_path)[1]}"
    shutil.move(img_path, os.path.join(train_fake_dir, new_name))

print(f"Moving {len(val_fake_images)} fake images to validation.")
for i, img_path in enumerate(val_fake_images):
    new_name = f"fake_{i:05d}{os.path.splitext(img_path)[1]}"
    shutil.move(img_path, os.path.join(val_fake_dir, new_name))

print("Fake images processed.")

Found 50000 fake images.
Moving 40000 fake images to training.
Moving 10000 fake images to validation.
Fake images processed.


In [ ]:
# Remove the original downloaded data directory to save space
if os.path.exists("./kaggle_data"):
    shutil.rmtree("./kaggle_data")
    print("Removed original ./kaggle_data directory.")

print("\nVerifying final directory structure and counts:")
!ls -Fh {dataset_dir}
!echo ""
!ls -Fh {train_real_dir} | wc -l
!ls -Fh {train_fake_dir} | wc -l
!ls -Fh {val_real_dir} | wc -l
!ls -Fh {val_fake_dir} | wc -l

Removed original ./kaggle_data directory.

Verifying final directory structure and counts:
train/	val/

40000
40000
10000
10000


# **Model creation and training part**


sanity check

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.backends.cudnn.version())

True
12.8
91900


In [ ]:
import torch
import csv
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import os
import numpy as np
import random
from google.colab import drive

# ============================================================
# MOUNT DRIVE
# ============================================================
drive.mount('/content/drive', force_remount=True)

# ============================================================
# CONFIGURATION
# ============================================================
PROJECT_ROOT = "/content/Faces_Project"
DATASET_ROOT = f"{PROJECT_ROOT}/Dataset"
CHECKPOINT_DIR = "/content/drive/MyDrive/Faces_Project/patch_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# MODEL SETTINGS
IMAGE_SIZE = 512
PATCH_SIZE = 112
OVERLAP = 20
BATCH_SIZE = 16  # Smaller because we process ALL patches
EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4  # Changed from 4 to 2 to avoid warning
SEED = 42

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Patch size: {PATCH_SIZE}x{PATCH_SIZE}")
print(f"Overlap: {OVERLAP}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")

# ============================================================
# FIXED SEED
# ============================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ============================================================
# MODEL - 1.2M Parameter Patch CNN
# ============================================================
class PatchCNN(nn.Module):
    """
    1.2M parameter CNN optimized for 112x112 patches
    """
    def __init__(self):
        super().__init__()

        # Feature Extractor (~900K params)
        self.features = nn.Sequential(
            # Block 1: 112 -> 56
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 56 -> 28
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: 28 -> 14
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4: 14 -> 7
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        # Classifier (~300K params)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ============================================================
# COUNT PARAMETERS
# ============================================================
model = PatchCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 60)
print("MODEL")
print("=" * 60)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# ============================================================
# PATCH EXTRACTION (EXHAUSTIVE + EDGE COVERAGE)
# ============================================================
def extract_patches(image, patch_size=PATCH_SIZE, overlap=OVERLAP):
    """
    Extract ALL overlapping patches with full edge coverage.
    Returns: List of [batch, 3, patch_size, patch_size]
    """
    _, _, H, W = image.shape
    stride = patch_size - overlap

    # Calculate positions ensuring edges are covered
    positions = []
    pos = 0
    while pos + patch_size <= H:
        positions.append(pos)
        if pos + patch_size == H:
            break
        pos += stride
        if pos + patch_size > H:
            positions.append(H - patch_size)
            break

    patches = []
    for y in positions:
        for x in positions:
            patch = image[:, :, y:y+patch_size, x:x+patch_size]
            patches.append(patch)

    return patches

# Test patch extraction
test_image = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
num_patches = len(extract_patches(test_image))
print(f"\nPatches per image: {num_patches} ({int(num_patches**0.5)}x{int(num_patches**0.5)} grid)")

# ============================================================
# EXTREME ENSEMBLE VOTING (FIXED - TENSOR OPERATIONS)
# ============================================================
def extreme_ensemble(patch_probs_tensor):
    """
    Takes patch probabilities as a tensor [num_patches, batch_size, 1]
    Returns: Final probability tensor with gradient flow

    Strategy: Simple average of ALL patches (stable for training)
    """
    # patch_probs_tensor shape: [num_patches, batch_size, 1]
    # Average across all patches (dim=0)
    final_pred = patch_probs_tensor.mean(dim=0)  # [batch_size, 1]

    return final_pred

# ============================================================
# DATASET TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# ============================================================
# DATASETS
# ============================================================
print("\n" + "=" * 60)
print("DATASET")
print("=" * 60)

train_dataset = datasets.ImageFolder(
    root=f"{DATASET_ROOT}/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=f"{DATASET_ROOT}/val",
    transform=val_transform
)

print(f"Classes: {train_dataset.classes}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

# ============================================================
# DATALOADERS
# ============================================================
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

# ============================================================
# LOSS, OPTIMIZER, SCHEDULER
# ============================================================
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

# ============================================================
# CHECKPOINT FUNCTIONS
# ============================================================
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "latest_patch_checkpoint.pth")
METRICS_PATH = os.path.join(CHECKPOINT_DIR, "patch_training_metrics.csv")

def save_checkpoint(epoch, model, optimizer, scheduler, scaler,
                    best_val_acc, best_f1, val_loss, rng_states, is_best=False):
    """Saves everything needed to resume training."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler else None,
        'best_val_acc': best_val_acc,
        'best_f1': best_f1,
        'last_val_loss': val_loss,
        'rng_states': rng_states,
    }
    # Always save the latest
    torch.save(checkpoint, CHECKPOINT_PATH)

    # Save best model separately if requested
    if is_best:
        torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, "best_patch_checkpoint.pth"))
        print(f"   >>> Best checkpoint saved! (Acc: {best_val_acc:.2f}%)")

def load_checkpoint(model, optimizer, scheduler, scaler):
    """Loads checkpoint and returns starting epoch."""
    if not os.path.exists(CHECKPOINT_PATH):
        print("\n>>> No checkpoint found. Starting fresh training from epoch 0.")
        return 0, 0.0, 0.0, 0.0

    try:
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    except Exception as e:
        print(f"⚠️ Falling back to weights_only=False (trusted source)")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint.get('best_val_acc', 0.0)
    best_f1 = checkpoint.get('best_f1', 0.0)
    last_val_loss = checkpoint.get('last_val_loss', 0.0)

    print(f"\n>>> Resuming from epoch {start_epoch}")
    print(f"   Best val acc so far: {best_val_acc:.2f}%")
    print(f"   Last val loss: {last_val_loss:.4f}")

    return start_epoch, best_val_acc, best_f1, last_val_loss

def log_metrics(epoch, train_loss, train_acc, val_acc, precision, recall, f1, lr):
    """Appends epoch metrics to a CSV file."""
    file_exists = os.path.exists(METRICS_PATH)

    with open(METRICS_PATH, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['epoch', 'train_loss', 'train_acc', 'val_acc',
                             'precision', 'recall', 'f1', 'learning_rate'])
        writer.writerow([epoch+1, f"{train_loss:.4f}", f"{train_acc:.2f}",
                         f"{val_acc:.2f}", f"{precision:.4f}", f"{recall:.4f}",
                         f"{f1:.4f}", f"{lr:.7f}"])

# ============================================================
# LOAD CHECKPOINT (BEFORE TRAINING)
# ============================================================
start_epoch, best_val_acc, best_f1, last_val_loss = load_checkpoint(
    model, optimizer, scheduler, scaler
)

if start_epoch > 0:
    print(f"   Resuming from epoch {start_epoch}")
    print(f"   Current LR: {optimizer.param_groups[0]['lr']:.7f}")
else:
    print("\n>>> Starting fresh training from epoch 0.")

# ============================================================
# TRAINING LOOP
# ============================================================
print("\n" + "=" * 60)
print("TRAINING STARTED")
print("=" * 60)

best_epoch = start_epoch
print(best_epoch)
for epoch in range(start_epoch,EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        # Extract ALL patches
        patches = extract_patches(images)  # 36 patches

        # Process each patch
        batch_patch_preds = []
        for patch in patches:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                pred = model(patch)  # [batch, 1]
                batch_patch_preds.append(pred)

        # Stack all predictions: [36, batch, 1]
        all_preds = torch.stack(batch_patch_preds)  # [36, batch, 1]
        all_preds = all_preds.squeeze(-1)  # [36, batch]

        # ✅ CORRECT: Flatten for loss
        all_preds_flat = all_preds.reshape(-1, 1)  # [36 * batch, 1]
        labels_flat = labels.repeat(36, 1)  # [36 * batch, 1]

        # Compute loss on ALL patches independently
        loss = criterion(all_preds_flat, labels_flat)

        # Backward pass
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        # Calculate accuracy (on all patches)
        predictions = (all_preds_flat >= 0.5).float()
        train_correct += (predictions == labels_flat).sum().item()
        train_total += labels_flat.size(0)

        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_loss = train_loss / len(train_loader)
    train_acc = 100 * train_correct / train_total

    # ============================================================
    # VALIDATION PHASE (SAME LOGIC)
    # ============================================================
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    TP = TN = FP = FN = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

            patches = extract_patches(images)

            batch_patch_preds = []
            for patch in patches:
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                    pred = model(patch)
                    batch_patch_preds.append(pred)

            all_preds = torch.stack(batch_patch_preds)  # [36, batch, 1]
            all_preds = all_preds.squeeze(-1)  # [36, batch]

            all_preds_flat = all_preds.reshape(-1, 1)  # [36 * batch, 1]
            labels_flat = labels.repeat(36, 1)  # [36 * batch, 1]

            loss = criterion(all_preds_flat, labels_flat)
            val_loss += loss.item()

            predictions = (all_preds_flat >= 0.5).float()

            val_correct += (predictions == labels_flat).sum().item()
            val_total += labels_flat.size(0)

            TP += ((predictions == 1) & (labels_flat == 1)).sum().item()
            TN += ((predictions == 0) & (labels_flat == 0)).sum().item()
            FP += ((predictions == 1) & (labels_flat == 0)).sum().item()
            FN += ((predictions == 0) & (labels_flat == 1)).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total
    precision = TP / (TP + FP) if TP + FP > 0 else 0.0
    recall = TP / (TP + FN) if TP + FN > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    # Save best model
    is_best = False
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        is_best = True
        torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/best_patch_model.pth")
        print(f"   >>> Best model saved! (Acc: {best_val_acc:.2f}%)")

    rng_states = {
        'random': random.getstate(),
        'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        rng_states['cuda'] = torch.cuda.get_rng_state()

    save_checkpoint(
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        best_val_acc=best_val_acc,
        best_f1=best_f1,
        val_loss=val_loss,
        rng_states=rng_states,
        is_best=is_best
    )

    log_metrics(epoch, train_loss, train_acc, val_acc, precision, recall, f1, current_lr)

    print("\n" + "=" * 60)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss:        {train_loss:.4f}")
    print(f"Train Accuracy:    {train_acc:.2f}%")
    print(f"Validation Loss:   {val_loss:.4f}")
    print(f"Validation Acc:    {val_acc:.2f}%")
    print(f"Precision:         {precision:.4f}")
    print(f"Recall:            {recall:.4f}")
    print(f"F1 Score:          {f1:.4f}")
    print(f"Learning Rate:     {current_lr:.7f}")
    print(f"Best Val Acc so far: {best_val_acc:.2f}% (Epoch {best_epoch})")
    if is_best:
        print(f"🎉 NEW BEST MODEL! 🎉")

# ============================================================
# TRAINING COMPLETE
# ============================================================
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Best validation accuracy: {best_val_acc:.2f}% (Epoch {best_epoch})")
print(f"Best model saved to: {CHECKPOINT_DIR}/best_patch_model.pth")

# ============================================================
# QUICK INFERENCE TEST
# ============================================================
print("\n" + "=" * 60)
print("QUICK INFERENCE TEST")
print("=" * 60)

model.eval()
test_image, test_label = next(iter(val_loader))
test_image = test_image[:1].to(device)
test_label = test_label[:1].to(device)

patches = extract_patches(test_image)
patch_probs = []
for patch in patches:
    with torch.no_grad():
        pred = model(patch)
        prob = torch.sigmoid(pred).item()
        patch_probs.append(prob)

# Show patch probabilities (sorted for clarity)
print(f"Patch probabilities (sorted): {[f'{p:.3f}' for p in sorted(patch_probs)]}")

# Apply extreme ensemble (using the tensor version with a dummy batch dimension)
patch_probs_tensor = torch.tensor(patch_probs, device=device).unsqueeze(1).unsqueeze(1)  # [num_patches, 1, 1]
final_prob_tensor = extreme_ensemble(patch_probs_tensor)  # [1, 1]
final_prob = final_prob_tensor.squeeze().item()

pred_label = "FAKE" if final_prob >= 0.5 else "REAL"
true_label = "FAKE" if test_label.item() == 1 else "REAL"

print(f"True label: {true_label}")
print(f"Predicted: {pred_label} (Confidence: {final_prob:.2%})")
print(f"Number of patches: {len(patches)}")
print("\n✅ Model is ready for deployment!")

# ============================================================
# SAVE FINAL MODEL
# ============================================================
torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/final_patch_model.pth")
print(f"Final model saved to: {CHECKPOINT_DIR}/final_patch_model.pth")

Mounted at /content/drive
CONFIGURATION
Image size: 512x512
Patch size: 112x112
Overlap: 20
Batch size: 16
Epochs: 25

Device: cuda
GPU: Tesla T4
VRAM: 15.64 GB

MODEL
Total parameters: 1,215,393
Trainable parameters: 1,215,393

Patches per image: 36 (6x6 grid)

DATASET
Classes: ['fake', 'real']
Training images: 80000
Validation images: 20000


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


⚠️ Falling back to weights_only=False (trusted source)

>>> Resuming from epoch 3
   Best val acc so far: 97.34%
   Last val loss: 0.0612
   Resuming from epoch 3
   Current LR: 0.0000965

TRAINING STARTED
3


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 4/25 | Batch 50/5000 | Loss: 0.0887
Epoch 4/25 | Batch 100/5000 | Loss: 0.0555
Epoch 4/25 | Batch 150/5000 | Loss: 0.1082
Epoch 4/25 | Batch 200/5000 | Loss: 0.0951
Epoch 4/25 | Batch 250/5000 | Loss: 0.0930
Epoch 4/25 | Batch 300/5000 | Loss: 0.0666
Epoch 4/25 | Batch 350/5000 | Loss: 0.0591
Epoch 4/25 | Batch 400/5000 | Loss: 0.0776
Epoch 4/25 | Batch 450/5000 | Loss: 0.0492
Epoch 4/25 | Batch 500/5000 | Loss: 0.1166
Epoch 4/25 | Batch 550/5000 | Loss: 0.0795
Epoch 4/25 | Batch 600/5000 | Loss: 0.2518
Epoch 4/25 | Batch 650/5000 | Loss: 0.1032
Epoch 4/25 | Batch 700/5000 | Loss: 0.2624
Epoch 4/25 | Batch 750/5000 | Loss: 0.1337
Epoch 4/25 | Batch 800/5000 | Loss: 0.0521
Epoch 4/25 | Batch 850/5000 | Loss: 0.0355
Epoch 4/25 | Batch 900/5000 | Loss: 0.0238
Epoch 4/25 | Batch 950/5000 | Loss: 0.0876
Epoch 4/25 | Batch 1000/5000 | Loss: 0.0595
Epoch 4/25 | Batch 1050/5000 | Loss: 0.0402
Epoch 4/25 | Batch 1100/5000 | Loss: 0.0588
Epoch 4/25 | Batch 1150/5000 | Loss: 0.1063
Epoch 4/